In [3]:
from pyspark.sql.functions import (
    col, year, month, quarter, dayofmonth, date_format, dayofweek, when, current_timestamp
)


StatementMeta(, 1905fff1-3bca-45ed-aaec-d665e6ab153f, 5, Finished, Available, Finished, False)

In [4]:
# 1. DIMENSION: Dim_User (Demographics & Segments)
# ==============================================================================
df_silver_users = spark.read.table("silver_users")

gold_dim_user = (
    df_silver_users.select(
        "user_id",
        "hashed_email",
        "user_name",
        "age",
        "gender",
        "marital_status",
        "occupation",
        "monthly_income",
        "educational_qualifications",
        "family_size"
    )
    .withColumn("created_timestamp", current_timestamp())
)

gold_dim_user.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_user")

StatementMeta(, a013799f-9d63-4596-9e3f-d997f2721b12, 6, Finished, Available, Finished, False)

In [4]:
# 2. DIMENSION: Dim_Restaurant (Partner Details & Location)
# ==============================================================================
df_silver_restaurant = spark.read.table("silver_restaurant")

gold_dim_restaurant = (
    df_silver_restaurant.select(
        "restaurant_id",
        "restaurant_name",
        "city",
        "rating",
        "rating_count",
        "cost",
        "cuisine",
        "lic_no",
        "address"
    )
    .withColumn("created_timestamp", current_timestamp())
)

gold_dim_restaurant.write.format("delta").mode("overwrite").saveAsTable("gold_dim_restaurant")

StatementMeta(, ef55ec38-82de-4982-a933-99d820ca2ad0, 6, Finished, Available, Finished, False)

In [5]:
# 3. DIMENSION: Dim_Food (Catalog Items)
# ==============================================================================
df_silver_food = spark.read.table("silver_food")

gold_dim_food = (
    df_silver_food.select(
        "food_id",
        "item_name",
        "veg_or_non_veg"
    )
    .withColumn("created_timestamp", current_timestamp())
)

gold_dim_food.write.format("delta").mode("overwrite").saveAsTable("gold_dim_food")

StatementMeta(, a013799f-9d63-4596-9e3f-d997f2721b12, 7, Finished, Available, Finished, False)

In [4]:
# 4. DIMENSION: Dim_Date (Time Intelligence for Power BI)
# ==============================================================================
df_silver_orders = spark.read.table("silver_orders")

# Extract unique order dates to build a dynamic date calendar table
df_dates = df_silver_orders.select("order_date").distinct().filter(col("order_date").isNotNull())

gold_dim_date = (
    df_dates.select(
        col("order_date").alias("date_key"),
        year("order_date").alias("year"),
        quarter("order_date").alias("quarter"),
        month("order_date").alias("month"),
        date_format("order_date", "MMMM").alias("month_name"),
        dayofmonth("order_date").alias("day_of_month"),
        date_format("order_date", "EEEE").alias("day_name"),
        when(dayofweek("order_date").isin(1, 7), True).otherwise(False).alias("is_weekend")
    )
)

gold_dim_date.write.format("delta").mode("overwrite").saveAsTable("gold_dim_date")

StatementMeta(, 1905fff1-3bca-45ed-aaec-d665e6ab153f, 6, Finished, Available, Finished, False)

In [6]:
# 5. FACT: Fact_Orders (Header Level Financial Metrics & Performance)
# ==============================================================================
gold_fact_orders = (
    df_silver_orders.select(
        "order_id",
        "user_id",
        col("restaurant_id").cast("string").alias("restaurant_id"),
        col("order_date").alias("date_key"),
        "order_timestamp",
        "restaurant_city",
        "cuisine",
        "payment_method",
        "order_status",
        "items_count",
        "sales_qty",
        "subtotal",
        "discount",
        "delivery_fee",
        "gst",
        "sales_amount",
        "customer_rating",
        "delivery_time_min"
    )
    .withColumn("created_timestamp", current_timestamp())
)

gold_fact_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_fact_orders")

StatementMeta(, 1905fff1-3bca-45ed-aaec-d665e6ab153f, 8, Finished, Available, Finished, False)

In [8]:
# 6. FACT: Fact_Order_Items (Line-Item Granularity Revenue Analysis)
# ==============================================================================
df_silver_order_items = spark.read.table("silver_order_items")

gold_fact_order_items = (
    df_silver_order_items.select(
        "order_item_id",
        "order_id",
        "restaurant_id",
        "food_id",
        "price",
        "quantity",
        "line_amount"
    )
    .withColumn("created_timestamp", current_timestamp())
)

gold_fact_order_items.write.format("delta").mode("overwrite").saveAsTable("gold_fact_order_items")

StatementMeta(, a013799f-9d63-4596-9e3f-d997f2721b12, 10, Finished, Available, Finished, True)